In [ ]:
import logging
import os
import sys
sys.path.append("../")
import glob
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import monai
from monai.data import ImageDataset, DataLoader
import monai.transforms as transforms
from monai.transforms import EnsureChannelFirst, Compose, RandRotate90, Resize, ScaleIntensity
import nibabel as nib
import pandas as pd

from utils.custom_transforms import ScaleIntensityFromHistogramPeak, SetBackgroundToZero, SelectChannelsd

In [ ]:
#ROOT_DIR = "/home/fehrdelt/bettik/"
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

In [ ]:
SUB_EXPERIMENT_NAME = "densenet3d_exp_0_0"

### Make the images used for classification
Stacking the T2 FLAIR, the anomaly map and the atlas together

In [ ]:
flair_timesteps = 130
adc_timesteps = 90

# SOOP

In [ ]:
anomaly_maps_flair_dir = ROOT_DIR+"datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/"
flair_dir = ROOT_DIR+"datasets/final_soop_dataset_small/flair_registered/"

anomaly_maps_adc_dir = ROOT_DIR+"datasets/anomaly_maps/exp_2_2_fixed_select_params/medium/"
adc_dir = ROOT_DIR+"datasets/final_soop_dataset_small/adc_registered/"

registered_atlases_dir = ROOT_DIR+"datasets/final_soop_dataset_small/registered_atlases/"

anomaly_maps_flair_paths = glob.glob(anomaly_maps_flair_dir+f"*{flair_timesteps}.nii.gz")
flair_paths = glob.glob(flair_dir+"*.nii.gz")

anomaly_maps_adc_paths = glob.glob(anomaly_maps_adc_dir+f"*{adc_timesteps}.nii.gz")
adc_paths = glob.glob(adc_dir+"*.nii.gz")

registered_atlases_paths = glob.glob(registered_atlases_dir+"*.nii.gz")

In [ ]:
print(anomaly_maps_flair_paths[:5])

['/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1608_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-123_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1296_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1055_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1081_t_130.nii.gz']


In [ ]:
""" for file in os.listdir(registered_atlases_dir):
    #rename files to have consistent naming
    if "sub-" in file:
        new_file_name = file.replace("_T1w", "")
        os.rename(os.path.join(registered_atlases_dir, file), os.path.join(registered_atlases_dir, new_file_name)) """

' for file in os.listdir(registered_atlases_dir):\n    #rename files to have consistent naming\n    if "sub-" in file:\n        new_file_name = file.replace("_T1w", "")\n        os.rename(os.path.join(registered_atlases_dir, file), os.path.join(registered_atlases_dir, new_file_name)) '

In [ ]:
# read the csv to get files to exclude
exclude_csv_path = ROOT_DIR+"StrokeUADiag/data_splits_lists/soop/exclude_failed_registration.csv"
exclude_df = pd.read_csv(exclude_csv_path, header=None)



exclude_files = exclude_df[0].tolist()

print(exclude_files)

['sub-185', 'sub_1303', 'sub-199', 'sub-984', 'sub-1138', 'sub-767', 'sub-1251', 'sub-855', 'sub-1660', 'sub-512', 'sub-1698', 'sub-617', 'sub-1119', 'sub-1183', 'sub-1558', 'sub-279', 'sub-846', 'sub-1610', 'sub-1261', 'sub-1308', 'sub-1717', 'sub-7', 'sub-1041', 'sub-343', 'sub-989', 'sub-605', 'sub-234', 'sub-1203', 'sub-1491', 'sub-949', 'sub-1727']


### Make stacked images (flair image, anomaly map & registered atlas)

In [ ]:
def normalize_from_histogram_peak(image, hist_norm_target_value=200.0):
    hist, bins = np.histogram(image, bins=100, range=(np.max(image)/15.0, np.max(image)*0.8))

    # Find the value corresponding to the maximum of the histogram
    most_occurred_pixel_value = bins[np.argmax(hist)]

    image_norm = image/most_occurred_pixel_value*hist_norm_target_value # scale it so the peak is always at hist_norm_target_value
    
    return image_norm

In [ ]:
# make the directory
os.makedirs(ROOT_DIR+f"datasets/StrokeUADiag_classification_inputs/", exist_ok=True)

In [ ]:

for ano_flair_map_path in tqdm(anomaly_maps_flair_paths):

    id = os.path.basename(ano_flair_map_path).replace(".nii.gz", "").split('_')[0]
    
    

    if id not in exclude_files:

        flair_img_path = f"{flair_dir}{id}.nii.gz"
        adc_img_path = f"{adc_dir}{id}.nii.gz"
        ano_adc_map_path = f"{anomaly_maps_adc_dir}{id}_t_{adc_timesteps}.nii.gz"
        registered_atlas_path = f"{registered_atlases_dir}{id}.nii.gz"

        output_path = f"{ROOT_DIR}datasets/StrokeUADiag_classification_inputs/medium/select_params/stacked_{id}.nii.gz"

        if os.path.exists(output_path):
            print(f"Stacked image for ID {id} already exists. Skipping.")
            continue

        # flair
        try:
            flair_nii = nib.load(flair_img_path)
            flair_image = normalize_from_histogram_peak(flair_nii.get_fdata(), hist_norm_target_value=200.0)
        except Exception as e:
            print(f"Error loading FLAIR image for ID {id}: {e}")
            continue
        try:
            ano_map_flair_image = nib.load(ano_flair_map_path).get_fdata()*1800.0
        except Exception as e:
            print(f"Error loading FLAIR anomaly map for ID {id}: {e}")
            continue

        # adc
        try:
            adc_nii = nib.load(adc_img_path)
            adc_image = normalize_from_histogram_peak(adc_nii.get_fdata(), hist_norm_target_value=200.0)
        except Exception as e:
            print(f"Error loading ADC image for ID {id}: {e}")
            continue
        try:
            ano_map_adc_image = nib.load(ano_adc_map_path).get_fdata()*1800.0
        except Exception as e:
            print(f"Error loading ADC anomaly map for ID {id}: {e}")
            continue

        # atlas
        try:
            registered_atlas_image = nib.load(registered_atlas_path).get_fdata()*10.0
        except Exception as e:
            print(f"Error loading registered atlas image for ID {id}: {e}")
            continue

        stacked_data = np.stack([flair_image, ano_map_flair_image, adc_image, ano_map_adc_image, registered_atlas_image], axis=-1)
        np.clip(stacked_data, 0, 1000, out=stacked_data)  # clip values to [0, 1000] to avoid extreme outliers
        stacked_img = nib.Nifti1Image(stacked_data, affine=flair_nii.affine)

        nib.save(stacked_img, output_path)
    
     

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 112/112 [05:31<00:00,  2.96s/it]


# AINI-Stroke

In [ ]:
anomaly_maps_aini_stroke_flair_dir = ROOT_DIR+"datasets/anomaly_maps//"
aini_stroke_flair_dir = ROOT_DIR+"datasets/final_aini_stroke_dataset_small/flair_registered/"

anomaly_maps_aini_stroke_adc_dir = ROOT_DIR+"datasets/anomaly_maps//"
aini_stroke_adc_dir = ROOT_DIR+"datasets/final_aini_stroke_dataset_small/adc_registered/"

registered_atlases_dir = ROOT_DIR+"datasets/final_aini_stroke_dataset_small/registered_atlases/"

anomaly_maps_flair_paths = glob.glob(anomaly_maps_aini_stroke_flair_dir+f"*{flair_timesteps}.nii.gz")
flair_paths = glob.glob(aini_stroke_flair_dir+"*.nii.gz")

anomaly_maps_adc_paths = glob.glob(anomaly_maps_adc_dir+f"*{adc_timesteps}.nii.gz")
adc_paths = glob.glob(aini_stroke_adc_dir+"*.nii.gz")

registered_atlases_paths = glob.glob(registered_atlases_dir+"*.nii.gz")

In [ ]:
print(anomaly_maps_flair_paths[:5])

['/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1608_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-123_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1296_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1055_t_130.nii.gz', '/bettik/PROJECTS/pr-gin5_aini/fehrdelt/datasets/anomaly_maps/exp_3_2_fixed_select_params/medium/sub-1081_t_130.nii.gz']


In [ ]:
""" for file in os.listdir(registered_atlases_dir):
    #rename files to have consistent naming
    if "sub-" in file:
        new_file_name = file.replace("_T1w", "")
        os.rename(os.path.join(registered_atlases_dir, file), os.path.join(registered_atlases_dir, new_file_name)) """

' for file in os.listdir(registered_atlases_dir):\n    #rename files to have consistent naming\n    if "sub-" in file:\n        new_file_name = file.replace("_T1w", "")\n        os.rename(os.path.join(registered_atlases_dir, file), os.path.join(registered_atlases_dir, new_file_name)) '

In [ ]:
# read the csv to get files to exclude
exclude_csv_path = ROOT_DIR+"StrokeUADiag/data_splits_lists/soop/exclude_failed_registration.csv"
exclude_df = pd.read_csv(exclude_csv_path, header=None)



exclude_files = exclude_df[0].tolist()

print(exclude_files)

['sub-185', 'sub_1303', 'sub-199', 'sub-984', 'sub-1138', 'sub-767', 'sub-1251', 'sub-855', 'sub-1660', 'sub-512', 'sub-1698', 'sub-617', 'sub-1119', 'sub-1183', 'sub-1558', 'sub-279', 'sub-846', 'sub-1610', 'sub-1261', 'sub-1308', 'sub-1717', 'sub-7', 'sub-1041', 'sub-343', 'sub-989', 'sub-605', 'sub-234', 'sub-1203', 'sub-1491', 'sub-949', 'sub-1727']


### Make stacked images (flair image, anomaly map & registered atlas)

In [ ]:
def normalize_from_histogram_peak(image, hist_norm_target_value=200.0):
    hist, bins = np.histogram(image, bins=100, range=(np.max(image)/15.0, np.max(image)*0.8))

    # Find the value corresponding to the maximum of the histogram
    most_occurred_pixel_value = bins[np.argmax(hist)]

    image_norm = image/most_occurred_pixel_value*hist_norm_target_value # scale it so the peak is always at hist_norm_target_value
    
    return image_norm

In [ ]:
# make the directory
os.makedirs(ROOT_DIR+f"datasets/StrokeUADiag_classification_inputs/", exist_ok=True)

In [ ]:

for ano_flair_map_path in tqdm(anomaly_maps_flair_paths):

    id = os.path.basename(ano_flair_map_path).replace(".nii.gz", "").split('_')[0]
    
    

    if id not in exclude_files:

        flair_img_path = f"{flair_dir}{id}.nii.gz"
        adc_img_path = f"{adc_dir}{id}.nii.gz"
        ano_adc_map_path = f"{anomaly_maps_adc_dir}{id}_t_{adc_timesteps}.nii.gz"
        registered_atlas_path = f"{registered_atlases_dir}{id}.nii.gz"

        output_path = f"{ROOT_DIR}datasets/StrokeUADiag_classification_inputs/medium/select_params/stacked_{id}.nii.gz"

        if os.path.exists(output_path):
            print(f"Stacked image for ID {id} already exists. Skipping.")
            continue

        # flair
        try:
            flair_nii = nib.load(flair_img_path)
            flair_image = normalize_from_histogram_peak(flair_nii.get_fdata(), hist_norm_target_value=200.0)
        except Exception as e:
            print(f"Error loading FLAIR image for ID {id}: {e}")
            continue
        try:
            ano_map_flair_image = nib.load(ano_flair_map_path).get_fdata()*1800.0
        except Exception as e:
            print(f"Error loading FLAIR anomaly map for ID {id}: {e}")
            continue

        # adc
        try:
            adc_nii = nib.load(adc_img_path)
            adc_image = normalize_from_histogram_peak(adc_nii.get_fdata(), hist_norm_target_value=200.0)
        except Exception as e:
            print(f"Error loading ADC image for ID {id}: {e}")
            continue
        try:
            ano_map_adc_image = nib.load(ano_adc_map_path).get_fdata()*1800.0
        except Exception as e:
            print(f"Error loading ADC anomaly map for ID {id}: {e}")
            continue

        # atlas
        try:
            registered_atlas_image = nib.load(registered_atlas_path).get_fdata()*10.0
        except Exception as e:
            print(f"Error loading registered atlas image for ID {id}: {e}")
            continue

        stacked_data = np.stack([flair_image, ano_map_flair_image, adc_image, ano_map_adc_image, registered_atlas_image], axis=-1)
        np.clip(stacked_data, 0, 1000, out=stacked_data)  # clip values to [0, 1000] to avoid extreme outliers
        stacked_img = nib.Nifti1Image(stacked_data, affine=flair_nii.affine)

        nib.save(stacked_img, output_path)
    
     

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 112/112 [05:31<00:00,  2.96s/it]
